# Notebook 60R2: Clean-room reviewed-mask SAFE_REMOVE operationalisation

**Purpose**

Apply the frozen Notebook 59R1 REMOVE deployment bundle to Notebook 07 human-reviewed pages, using the reviewed mask as the sole intervention authority.

**Clean-room rules**

- No dependency on Notebooks 45, 46, 50, 51, 52, 53 or 54 at runtime.
- No import or execution of earlier Notebook 60 code.
- Notebook 07 aligned page + amended mask define the operational page and authority boundary.
- Notebook 06 confidence is never treated as `p_print`.
- `p_print` must come from a genuine observed-print probability artefact. This notebook accepts a precomputed page array or a separately frozen observed-print bundle.
- The Notebook 59R1 model, calibrator, schema, policy and hashes are verified before scoring.
- `SAFE_PROTECT` is disabled.
- Exact fallback is M3.
- Changes outside M3 are prohibited.
- This notebook is supervised experimental tooling, not automatic production authority.

In [ ]:
# 1. Imports, Drive mount and configuration
from pathlib import Path
from datetime import datetime, timezone
import hashlib, json, os, shutil, traceback

import cv2
import joblib
import numpy as np
import pandas as pd

try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception:
    pass

ROOT = Path('/content/drive/MyDrive/MusicAnnotationProject')

# Notebook 59R1 run root. Leave blank to select the newest complete run.
N59_ROOT = ROOT / 'remove_deployment_fit_freeze_v8_8_59r1'
N59_RUN_OVERRIDE = ''

# Notebook 07 audit directory.
REVIEW_AUDIT_DIR = ROOT / 'hitl_review' / 'audit'

# Genuine observed-print probabilities. One .npy/.npz file per canonical page.
# Supported names: PageXXX.npy, PageXXX_p_print.npy, PageXXX/p_print.npy.
P_PRINT_ROOT = ROOT / 'observed_print_runtime' / 'p_print'

# Optional separate all-development observed-print bundle. This clean-room notebook
# does not assume one exists. If supplied, its prediction adapter must be implemented
# explicitly in Cell 6 and must produce calibrated page-shaped probabilities.
OBSERVED_PRINT_BUNDLE_OVERRIDE = ''

OUTPUT_ROOT = ROOT / 'notebook60r2_clean_room'
RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S') + '_notebook60r2_clean_room'
OUT = OUTPUT_ROOT / RUN_ID
AUDIT = OUT / 'audit'
TABLES = OUT / 'tables'
PAGES = OUT / 'pages'
FAILURES = OUT / 'failures'
for p in [OUT, AUDIT, TABLES, PAGES, FAILURES]:
    p.mkdir(parents=True, exist_ok=True)

PAGE_ALLOWLIST = None
EXPECTED_REVIEWED_PAGES = 151
REQUIRE_EXACT_PAGE_COUNT = False
STOP_AT_FIRST_PAGE_FAILURE = False

print('Output:', OUT)


In [ ]:
# 2. General helpers

def require(condition, message):
    if not bool(condition):
        raise RuntimeError(message)

def read_json(path):
    path = Path(path)
    require(path.exists() and path.is_file(), f'Missing JSON: {path}')
    value = json.loads(path.read_text())
    require(isinstance(value, dict), f'JSON root must be an object: {path}')
    return value

def atomic_json(path, value):
    path = Path(path)
    tmp = path.with_suffix(path.suffix + '.tmp')
    tmp.write_text(json.dumps(value, indent=2, sort_keys=True, default=str))
    tmp.replace(path)

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for block in iter(lambda: f.read(1024 * 1024), b''):
            h.update(block)
    return h.hexdigest()

def array_sha256(value):
    value = np.ascontiguousarray(value)
    h = hashlib.sha256()
    h.update(str(value.dtype).encode())
    h.update(str(value.shape).encode())
    h.update(value.tobytes())
    return h.hexdigest()

def read_gray(path):
    path = Path(path)
    image = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
    require(image is not None, f'Could not read greyscale image: {path}')
    return image

def first_existing_path(record, names):
    for name in names:
        value = record.get(name)
        if value is None or not str(value).strip():
            continue
        path = Path(str(value).strip())
        if path.exists() and path.is_file():
            return path, name
    return None, None


In [ ]:
# 3. Resolve and verify one complete Notebook 59R1 deployment run

REQUIRED_BUNDLE = [
    'remove_model.joblib',
    'remove_calibrator.joblib',
    'predictor_schema.json',
    'policy.json',
    'reason_code_schema.json',
    'freeze_manifest.json',
    'runtime_contract.json',
]
REQUIRED_RUN = ['replay/replay_test.json']

def complete_59r1_runs(root):
    root = Path(root)
    if not root.exists():
        return []
    hits = []
    for run in root.iterdir():
        if not run.is_dir():
            continue
        bundle = run / 'bundle'
        if not bundle.is_dir():
            continue
        if all((bundle / name).is_file() for name in REQUIRED_BUNDLE) and all((run / name).is_file() for name in REQUIRED_RUN):
            hits.append(run)
    return sorted(hits, key=lambda p: p.stat().st_mtime)

if N59_RUN_OVERRIDE:
    N59 = Path(N59_RUN_OVERRIDE)
    if N59.name == 'bundle':
        N59 = N59.parent
else:
    hits = complete_59r1_runs(N59_ROOT)
    require(hits, f'No complete Notebook 59R1 run found below {N59_ROOT}')
    N59 = hits[-1]

BUNDLE = N59 / 'bundle'
REPLAY_PATH = N59 / 'replay' / 'replay_test.json'
missing = [str(BUNDLE / x) for x in REQUIRED_BUNDLE if not (BUNDLE / x).is_file()]
missing += [str(N59 / x) for x in REQUIRED_RUN if not (N59 / x).is_file()]
require(not missing, f'Notebook 59R1 run is incomplete: {missing}')

POLICY_PATH = BUNDLE / 'policy.json'
SCHEMA_PATH = BUNDLE / 'predictor_schema.json'
REASON_PATH = BUNDLE / 'reason_code_schema.json'
FREEZE_PATH = BUNDLE / 'freeze_manifest.json'
CONTRACT_PATH = BUNDLE / 'runtime_contract.json'
MODEL_PATH = BUNDLE / 'remove_model.joblib'
CALIBRATOR_PATH = BUNDLE / 'remove_calibrator.joblib'

policy = read_json(POLICY_PATH)
schema = read_json(SCHEMA_PATH)
reasons = read_json(REASON_PATH)
freeze = read_json(FREEZE_PATH)
contract = read_json(CONTRACT_PATH)
replay = read_json(REPLAY_PATH)

policy_id = policy.get('policy_id')
compatibility_id = policy.get('compatibility_id')
require(policy_id, '59R1 policy_id is missing.')
require(compatibility_id, '59R1 compatibility_id is missing.')
require(replay.get('passed') is True, '59R1 replay_test.json does not report passed=true.')
for name, obj in [('freeze', freeze), ('contract', contract), ('replay', replay)]:
    if obj.get('policy_id') is not None:
        require(obj.get('policy_id') == policy_id, f'{name} policy_id mismatch.')
    if obj.get('compatibility_id') is not None:
        require(obj.get('compatibility_id') == compatibility_id, f'{name} compatibility_id mismatch.')

# The freeze manifest may encode paths relative to bundle/. Verify every declared object.
for item in freeze.get('objects', []):
    rel = item.get('relative_path')
    expected = item.get('sha256')
    require(rel and expected, f'Invalid freeze object entry: {item}')
    target = BUNDLE / rel
    require(target.is_file(), f'Missing frozen object: {target}')
    require(sha256_file(target) == expected, f'Frozen object hash mismatch: {target}')

# Verify primary objects using policy hashes when present.
for key, path in [
    ('model_sha256', MODEL_PATH),
    ('calibrator_sha256', CALIBRATOR_PATH),
    ('predictor_schema_sha256', SCHEMA_PATH),
    ('reason_code_schema_sha256', REASON_PATH),
]:
    if policy.get(key):
        require(sha256_file(path) == policy[key], f'Policy hash mismatch for {path.name}')

predictors = schema.get('ordered_predictors') or schema.get('predictors') or schema.get('ordered_features')
require(isinstance(predictors, list) and len(predictors) == 17, '59R1 schema must contain exactly 17 ordered predictors.')
require(len(predictors) == len(set(predictors)), '59R1 predictor schema contains duplicates.')

model = joblib.load(MODEL_PATH)
calibrator = joblib.load(CALIBRATOR_PATH)
require(hasattr(model, 'predict_proba'), '59R1 model lacks predict_proba().')
require(hasattr(calibrator, 'predict'), '59R1 calibrator lacks predict().')
if hasattr(model, 'n_features_in_'):
    require(int(model.n_features_in_) == len(predictors), '59R1 model/schema feature-count mismatch.')

atomic_json(AUDIT / 'notebook59r1_source.json', {
    'run': str(N59), 'bundle': str(BUNDLE), 'replay_path': str(REPLAY_PATH),
    'policy_id': policy_id, 'compatibility_id': compatibility_id,
    'model_sha256': sha256_file(MODEL_PATH),
    'calibrator_sha256': sha256_file(CALIBRATOR_PATH),
    'schema_sha256': sha256_file(SCHEMA_PATH),
    'replay_sha256': sha256_file(REPLAY_PATH),
})
print('59R1 integrity: PASS')
print('Run:', N59)


In [ ]:
# 4. Resolve the Notebook 07 reviewed cohort only

records, unresolved = [], []
for audit_path in sorted(Path(REVIEW_AUDIT_DIR).glob('*.json')):
    try:
        review = read_json(audit_path)
        observed_path, observed_field = first_existing_path(review, [
            'aligned_output', 'aligned_page', 'observed_output', 'aligned_image', 'observed_path'
        ])
        mask_path, mask_field = first_existing_path(review, [
            'corrected_mask_output', 'corrected_mask', 'reviewed_mask_output', 'reviewed_mask', 'mask_output'
        ])
        page = str(review.get('page') or review.get('page_id') or review.get('page_name') or '').strip()
        if not page:
            page = observed_path.stem if observed_path is not None else audit_path.stem
        require(observed_path is not None, 'aligned observed page not resolved')
        require(mask_path is not None, 'reviewed mask not resolved')
        observed = read_gray(observed_path)
        reviewed = read_gray(mask_path)
        require(observed.shape == reviewed.shape, f'shape mismatch {observed.shape} vs {reviewed.shape}')
        require((reviewed > 0).any(), 'reviewed mask is empty')
        records.append({
            'page': page,
            'observed_path': str(observed_path),
            'reviewed_mask_path': str(mask_path),
            'review_audit_path': str(audit_path),
            'observed_sha256': sha256_file(observed_path),
            'reviewed_mask_sha256': sha256_file(mask_path),
            'height': observed.shape[0], 'width': observed.shape[1],
            'reviewed_mask_pixels': int((reviewed > 0).sum()),
        })
    except Exception as error:
        unresolved.append({'review_audit_path': str(audit_path), 'error': str(error)})

roster = pd.DataFrame(records)
pd.DataFrame(unresolved).to_csv(TABLES / 'unresolved_notebook07_reviews.csv', index=False)
require(not roster.empty, f'No valid Notebook 07 reviewed pages resolved from {REVIEW_AUDIT_DIR}')

# Identical duplicate audit records can be collapsed; conflicting identities block execution.
for page, group in roster.groupby('page'):
    require(group.observed_sha256.nunique() == 1 and group.reviewed_mask_sha256.nunique() == 1,
            f'Conflicting Notebook 07 records for {page}')
roster = roster.sort_values(['page', 'review_audit_path']).drop_duplicates('page').reset_index(drop=True)

if PAGE_ALLOWLIST is not None:
    wanted = {str(x) for x in PAGE_ALLOWLIST}
    missing = sorted(wanted - set(roster.page.astype(str)))
    require(not missing, f'Allowlisted pages unresolved: {missing}')
    roster = roster[roster.page.astype(str).isin(wanted)].copy()

if REQUIRE_EXACT_PAGE_COUNT:
    require(len(roster) == EXPECTED_REVIEWED_PAGES,
            f'Expected {EXPECTED_REVIEWED_PAGES} pages; resolved {len(roster)}')

roster.to_csv(TABLES / 'hitl_reviewed_roster.csv', index=False)
atomic_json(AUDIT / 'reviewed_roster_summary.json', {
    'review_audit_dir': str(REVIEW_AUDIT_DIR),
    'resolved_pages': len(roster), 'unresolved_records': len(unresolved),
    'mask_authority': 'Notebook 07 human-reviewed mask',
    'notebook06_confidence_used': False,
})
print('Reviewed pages:', len(roster))
display(roster.head())


In [ ]:
# 5. Canonical geometry, M3 preparation and 17 predictor construction

MASK_DILATION = int(policy.get('mask_dilation', 2))
CROP_MARGIN = int(policy.get('crop_margin', 48))
GEOMETRY_THRESHOLD = int(policy.get('geometry_threshold', 80))
BAND_WIDTH = int(policy.get('band_width', 4))
REMOVE_THRESHOLD = float(policy.get('remove_threshold', policy.get('safe_remove_threshold', 0.995)))
MIN_REMOVE_RUN = int(policy.get('minimum_remove_run', policy.get('minimum_safe_remove_run', 2)))
CONNECTIVITY = int(policy.get('connectivity', 8))
require(CONNECTIVITY in [4, 8], f'Unsupported connectivity: {CONNECTIVITY}')

def binary_mask(mask):
    return np.where(mask > 0, 255, 0).astype(np.uint8)

def prepare_mask(seed):
    mask = binary_mask(seed)
    if MASK_DILATION > 0:
        kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (2*MASK_DILATION+1, 2*MASK_DILATION+1))
        mask = cv2.dilate(mask, kernel)
    return mask

def immutable(original, candidate, authority):
    out = original.copy()
    out[authority > 0] = candidate[authority > 0]
    require(np.array_equal(out[authority == 0], original[authority == 0]), 'Outside-authority drift.')
    return out

def telea_reconstruct(observed, mask):
    candidate = cv2.inpaint(observed, binary_mask(mask), 3, cv2.INPAINT_TELEA)
    return immutable(observed, candidate, mask)

def background_surface(observed, mask, sigma=45.0):
    visible = (mask == 0) & (observed >= 200)
    num = cv2.GaussianBlur(observed.astype(np.float32) * visible, (0,0), sigma)
    den = cv2.GaussianBlur(visible.astype(np.float32), (0,0), sigma)
    fallback = cv2.GaussianBlur(observed, (0,0), sigma)
    return np.clip(np.where(den > 0.02, num / np.maximum(den, 1e-4), fallback), 0, 255).astype(np.uint8)

def support_maps(observed, mask):
    dark = (observed < 190) & (mask == 0)
    horizontal = cv2.morphologyEx(dark.astype(np.uint8)*255, cv2.MORPH_OPEN, np.ones((1,35),np.uint8)) > 0
    vertical = cv2.morphologyEx(dark.astype(np.uint8)*255, cv2.MORPH_OPEN, np.ones((15,1),np.uint8)) > 0
    n, labels, stats, _ = cv2.connectedComponentsWithStats(dark.astype(np.uint8), 8)
    compact = np.zeros_like(dark)
    for cid in range(1,n):
        x,y,w,h,area = stats[cid]
        if 3 <= area <= 650 and w <= 40 and h <= 40:
            compact[labels == cid] = True
    kernel = np.ones((2*BAND_WIDTH+1, 2*BAND_WIDTH+1), np.uint8)
    return tuple(cv2.dilate(x.astype(np.uint8), kernel) > 0 for x in (horizontal, vertical, compact))

def geometry_routes(observed, mask):
    paper = background_surface(observed, mask)
    normalised = cv2.divide(observed, np.maximum(paper,1), scale=255)
    horizontal, vertical, compact = support_maps(observed, mask)
    inside = mask > 0
    preserve = inside & (horizontal | vertical | compact) & (observed < 200)
    remove = inside & ~preserve & (observed >= GEOMETRY_THRESHOLD) & (normalised >= GEOMETRY_THRESHOLD)
    ambiguous = inside & ~preserve & ~remove
    return {'paper':paper, 'normalised':normalised, 'preserve':preserve, 'remove':remove,
            'ambiguous':ambiguous, 'horizontal':horizontal, 'vertical':vertical, 'compact':compact}

def local_fraction(mask, radius=3):
    k = 2*radius+1
    return cv2.boxFilter(mask.astype(np.float32), -1, (k,k), normalize=True, borderType=cv2.BORDER_REFLECT)

def distance_to_true(mask):
    mask = mask.astype(bool)
    if not mask.any():
        return np.full(mask.shape, float(max(mask.shape)), np.float32)
    return cv2.distanceTransform((~mask).astype(np.uint8), cv2.DIST_L2, 5).astype(np.float32)

def build_predictors(page, observed, m3, routes, p_print):
    inside = m3 > 0
    require(p_print.shape == observed.shape, f'{page}: p_print shape mismatch')
    edge = cv2.Canny(observed, 70, 170) > 0
    mask_depth = cv2.distanceTransform(inside.astype(np.uint8), cv2.DIST_L2, 5).astype(np.float32)
    eroded = cv2.erode(inside.astype(np.uint8), np.ones((3,3),np.uint8)) > 0
    boundary_distance = distance_to_true(inside ^ eroded)
    protected_distance = distance_to_true(routes['preserve'])
    _, component_id = cv2.connectedComponents(inside.astype(np.uint8), connectivity=8)
    y,x = np.where(inside)
    frame = pd.DataFrame({
        'page':page, 'y':y, 'x':x,
        'observed_value':observed[y,x].astype(np.float32),
        'normalised_value':routes['normalised'][y,x].astype(np.float32),
        'paper_value':routes['paper'][y,x].astype(np.float32),
        'mask_depth':mask_depth[y,x], 'boundary_distance':boundary_distance[y,x],
        'distance_to_protected':protected_distance[y,x], 'edge':edge[y,x].astype(np.float32),
        'local_edge_fraction':local_fraction(edge)[y,x],
        'local_dark_fraction':local_fraction(observed < 190)[y,x],
        'local_preserve_fraction':local_fraction(routes['preserve'])[y,x],
        'local_remove_fraction':local_fraction(routes['remove'])[y,x],
        'local_ambiguous_fraction':local_fraction(routes['ambiguous'])[y,x],
        'local_horizontal_fraction':local_fraction(routes['horizontal'])[y,x],
        'local_vertical_fraction':local_fraction(routes['vertical'])[y,x],
        'local_compact_fraction':local_fraction(routes['compact'])[y,x],
        'p_print':p_print[y,x].astype(np.float32),
        'component_id':component_id[y,x].astype(np.float32),
    })
    missing = [p for p in predictors if p not in frame.columns]
    require(not missing, f'{page}: missing predictors {missing}')
    X = frame.loc[:, predictors].apply(pd.to_numeric, errors='coerce')
    values = X.to_numpy(np.float64)
    require(not np.isinf(values).any(), f'{page}: infinite predictor value')
    return frame, X


In [ ]:
# 6. Genuine p_print resolver
#
# This notebook deliberately does not use Notebook 06 YOLO confidence.
# It accepts only a page-shaped observed-print probability array in [0,1].
# A separate all-development observed-print bundle may be added here later,
# but no OOF-fold averaging rule is invented in this clean-room version.

def p_print_candidates(page):
    return [
        P_PRINT_ROOT / f'{page}.npy',
        P_PRINT_ROOT / f'{page}_p_print.npy',
        P_PRINT_ROOT / f'{page}.npz',
        P_PRINT_ROOT / f'{page}_p_print.npz',
        P_PRINT_ROOT / page / 'p_print.npy',
        P_PRINT_ROOT / page / 'p_print.npz',
    ]

def load_p_print(page, expected_shape):
    matches = []
    seen = set()
    for path in p_print_candidates(page):
        if path.is_file():
            key = str(path.resolve())
            if key not in seen:
                seen.add(key); matches.append(path)
    require(len(matches) == 1,
            f'{page}: expected exactly one genuine p_print file under {P_PRINT_ROOT}; found {len(matches)}')
    path = matches[0]
    if path.suffix.lower() == '.npy':
        value = np.load(path, allow_pickle=False)
    else:
        archive = np.load(path, allow_pickle=False)
        keys = list(archive.files)
        preferred = [k for k in ['p_print','probability','probabilities','arr_0'] if k in keys]
        require(len(preferred) == 1, f'{page}: ambiguous npz keys {keys}')
        value = archive[preferred[0]]
    value = np.asarray(value)
    require(value.ndim == 2 and value.shape == expected_shape,
            f'{page}: p_print shape {value.shape} != {expected_shape}')
    value = value.astype(np.float32)
    require(np.isfinite(value).all(), f'{page}: non-finite p_print')
    require(((value >= 0) & (value <= 1)).all(), f'{page}: p_print outside [0,1]')
    return value, path

# Preflight without scoring or changing pages.
preflight, blocked = [], []
for row in roster.itertuples(index=False):
    try:
        observed = read_gray(row.observed_path)
        p_print, path = load_p_print(str(row.page), observed.shape)
        preflight.append({'page':row.page, 'p_print_path':str(path), 'p_print_sha256':sha256_file(path),
                          'min':float(p_print.min()), 'max':float(p_print.max()), 'mean':float(p_print.mean())})
    except Exception as error:
        blocked.append({'page':row.page, 'error':str(error)})

pd.DataFrame(preflight).to_csv(TABLES / 'p_print_preflight.csv', index=False)
pd.DataFrame(blocked).to_csv(TABLES / 'p_print_preflight_failures.csv', index=False)
require(not blocked,
        f'{len(blocked)} page(s) lack a valid genuine p_print array. See p_print_preflight_failures.csv')
print('p_print preflight: PASS for', len(preflight), 'pages')


In [ ]:
# 7. Frozen REMOVE scoring and region coherence

def positive_probability(estimator, X):
    matrix = np.asarray(estimator.predict_proba(X), dtype=np.float64)
    require(matrix.ndim == 2 and matrix.shape[0] == len(X) and matrix.shape[1] >= 2,
            f'Invalid predict_proba shape {matrix.shape}')
    classes = list(getattr(estimator, 'classes_', [0,1]))
    matches = [i for i,c in enumerate(classes) if c in [1, True, '1', 'REMOVE', 'SAFE_REMOVE'] or str(c).lower() in ['remove','safe_remove','true']]
    require(len(matches) == 1, f'Cannot identify positive REMOVE class from {classes}')
    return matrix[:, matches[0]]

def calibrated_remove_probability(X):
    raw = positive_probability(model, X)
    try:
        calibrated = np.asarray(calibrator.predict(raw), dtype=np.float64).reshape(-1)
    except Exception:
        calibrated = np.asarray(calibrator.predict(raw.reshape(-1,1)), dtype=np.float64).reshape(-1)
    require(calibrated.shape == raw.shape and np.isfinite(calibrated).all(), 'Invalid calibrated output')
    require(((calibrated >= 0) & (calibrated <= 1)).all(), 'Calibrated probability outside [0,1]')
    return raw, calibrated

def coherent_safe_remove(shape, y, x, probability):
    provisional = probability >= REMOVE_THRESHOLD
    canvas = np.zeros(shape, np.uint8)
    canvas[y[provisional], x[provisional]] = 1
    n, labels, stats, _ = cv2.connectedComponentsWithStats(canvas, connectivity=CONNECTIVITY)
    final = np.zeros(len(y), dtype=bool)
    reason = np.full(len(y), 'FALLBACK_M3', dtype=object)
    coordinate_labels = labels[y,x]
    for cid in range(1,n):
        idx = np.where((coordinate_labels == cid) & provisional)[0]
        if len(idx) == 0: continue
        valid = (int(stats[cid, cv2.CC_STAT_AREA]) >= MIN_REMOVE_RUN and
                 np.all(probability[idx] >= REMOVE_THRESHOLD) and
                 float(np.mean(probability[idx])) >= REMOVE_THRESHOLD)
        if valid:
            final[idx] = True; reason[idx] = 'SAFE_REMOVE'
        else:
            reason[idx] = 'REMOVE_REGION_REJECTED_FALLBACK_M3'
    return provisional, final, reason


In [ ]:
# 8. Batch execution

page_rows, failures, pixel_parts = [], [], []
for position, row in enumerate(roster.itertuples(index=False), start=1):
    page = str(row.page)
    try:
        observed = read_gray(row.observed_path)
        reviewed = read_gray(row.reviewed_mask_path)
        require(observed.shape == reviewed.shape, f'{page}: observed/mask shape mismatch')
        p_print, p_print_path = load_p_print(page, observed.shape)

        m3_u8 = prepare_mask(reviewed)
        m3 = m3_u8 > 0
        routes = geometry_routes(observed, m3_u8)
        frame, X = build_predictors(page, observed, m3_u8, routes, p_print)
        raw, probability = calibrated_remove_probability(X)
        y = frame.y.to_numpy(int); x = frame.x.to_numpy(int)
        provisional, decision, reason = coherent_safe_remove(observed.shape, y, x, probability)

        safe_remove = np.zeros(observed.shape, bool)
        safe_remove[y[decision], x[decision]] = True
        require(not np.any(safe_remove & ~m3), f'{page}: SAFE_REMOVE outside M3')
        m5 = m3 & ~safe_remove

        # Registered remove-only semantics: reconstruct M5; assign SAFE_REMOVE the frozen paper estimate.
        corrected = telea_reconstruct(observed, m5.astype(np.uint8)*255)
        corrected[safe_remove] = routes['paper'][safe_remove]
        require(np.array_equal(corrected[~m3], observed[~m3]), f'{page}: outside-M3 drift')

        page_dir = PAGES / page
        require(not page_dir.exists(), f'{page}: output directory already exists')
        page_dir.mkdir()
        outputs = {
            'observed.png': observed,
            'reviewed_seed.png': binary_mask(reviewed),
            'm3.png': m3.astype(np.uint8)*255,
            'safe_remove.png': safe_remove.astype(np.uint8)*255,
            'm5_remove.png': m5.astype(np.uint8)*255,
            'paper.png': routes['paper'],
            'corrected.png': corrected,
        }
        for name,value in outputs.items():
            require(cv2.imwrite(str(page_dir/name), value), f'{page}: failed to write {name}')

        predictions = frame[['page','y','x']].copy()
        predictions['raw_remove_score'] = raw
        predictions['p_remove'] = probability
        predictions['safe_remove_provisional'] = provisional
        predictions['safe_remove'] = decision
        predictions['reason'] = reason
        part = TABLES / f'{page}_predictions.parquet'
        predictions.to_parquet(part, index=False)
        pixel_parts.append(part)

        result = {
            'page': page, 'status':'PASS', 'm3_pixels':int(m3.sum()),
            'safe_remove_pixels':int(safe_remove.sum()), 'm5_pixels':int(m5.sum()),
            'outside_m3_drift':int(np.sum(corrected[~m3] != observed[~m3])),
            'm5_formula_xor':int(np.logical_xor(m5, m3 & ~safe_remove).sum()),
            'observed_sha256':array_sha256(observed), 'm3_sha256':array_sha256(m3.astype(np.uint8)),
            'safe_remove_sha256':array_sha256(safe_remove.astype(np.uint8)),
            'm5_sha256':array_sha256(m5.astype(np.uint8)), 'corrected_sha256':array_sha256(corrected),
            'p_print_path':str(p_print_path), 'p_print_sha256':sha256_file(p_print_path),
        }
        atomic_json(page_dir/'page_manifest.json', result)
        page_rows.append(result)
        print(f'[{position}/{len(roster)}] PASS {page}: SAFE_REMOVE={int(safe_remove.sum()):,}')
    except Exception as error:
        failure = {'page':page, 'status':'FAIL', 'error':str(error), 'traceback':traceback.format_exc()}
        failures.append(failure)
        atomic_json(FAILURES/f'{page}.json', failure)
        print(f'[{position}/{len(roster)}] FAIL {page}: {error}')
        if STOP_AT_FIRST_PAGE_FAILURE:
            raise

page_df = pd.DataFrame(page_rows)
failure_df = pd.DataFrame(failures)
page_df.to_csv(TABLES/'page_results.csv', index=False)
failure_df.to_csv(TABLES/'page_failures.csv', index=False)
if pixel_parts:
    pd.concat([pd.read_parquet(p) for p in pixel_parts], ignore_index=True).to_parquet(
        TABLES/'all_fresh_safe_remove_predictions.parquet', index=False)
print('Passed:', len(page_df), 'Failed:', len(failure_df))


In [ ]:
# 9. Integrity verdict and curated upload package

checks = {
    '59r1_replay_passed': replay.get('passed') is True,
    'all_roster_pages_accounted_for': len(page_df) + len(failure_df) == len(roster),
    'zero_page_failures': len(failure_df) == 0,
    'zero_outside_m3_drift': bool(len(page_df) and (page_df.outside_m3_drift == 0).all()),
    'zero_m5_formula_xor': bool(len(page_df) and (page_df.m5_formula_xor == 0).all()),
    'notebook06_confidence_used': False,
    'safe_protect_enabled': False,
    'production_authority': False,
}
first_failed_gate = next((k for k,v in checks.items() if not bool(v)), None)
verdict = 'PASS_60R2_CLEAN_ROOM_SUPERVISED' if first_failed_gate is None else 'FAIL_OR_PARTIAL_60R2_CLEAN_ROOM'
record = {
    'created_utc': datetime.now(timezone.utc).isoformat(),
    'version': '60R2-clean-room', 'verdict': verdict, 'first_failed_gate': first_failed_gate,
    'checks': checks, 'resolved_pages':len(roster), 'passed_pages':len(page_df), 'failed_pages':len(failure_df),
    'policy_id':policy_id, 'compatibility_id':compatibility_id,
    'notebook59_run':str(N59), 'review_audit_dir':str(REVIEW_AUDIT_DIR),
    'p_print_root':str(P_PRINT_ROOT), 'safe_protect_enabled':False,
    'automatic_production_authority':False,
}
atomic_json(OUT/'notebook60r2_verdict.json', record)

# Curated package deliberately excludes the two joblib files.
UPLOAD = OUT/'upload'
UPLOAD.mkdir(exist_ok=True)
for path in [OUT/'notebook60r2_verdict.json', AUDIT/'notebook59r1_source.json',
             AUDIT/'reviewed_roster_summary.json', TABLES/'hitl_reviewed_roster.csv',
             TABLES/'page_results.csv', TABLES/'page_failures.csv']:
    if path.exists(): shutil.copy2(path, UPLOAD/path.name)
manifest=[]
for path in sorted(UPLOAD.iterdir()):
    if path.is_file(): manifest.append({'file':path.name, 'bytes':path.stat().st_size, 'sha256':sha256_file(path)})
pd.DataFrame(manifest).to_csv(UPLOAD/'upload_manifest.csv', index=False)
archive = shutil.make_archive(str(OUT/'notebook60r2_upload'), 'zip', root_dir=UPLOAD)
print(json.dumps(record, indent=2))
print('Upload archive:', archive)
